# Editable Deformation Regime Composite

This notebook builds and saves the paper-artifact composite figure. Edit the controls in the next cell to change visible text, font sizes, layout, colors, output formats, selected methods, and crop bounds. Then rerun the cells below.

In [ ]:
from pathlib import Path

# Resolve the repository root from this notebook location.
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "experiments").exists():
    REPO_ROOT = REPO_ROOT.parent

ARTIFACT_DIR = REPO_ROOT / "experiments/paper_artifacts/deformation_regime_figure/v001"
MOTION_ARTIFACT_DIR = REPO_ROOT / "experiments/paper_artifacts/motion_pattern_figure/v001"
STATS_DIR = REPO_ROOT / "experiments/2026-06-25_motion_regime_seed3_first10_current_methods"

# All visible figure text lives here. Edit freely.
TEXT = {
    "panel_letter": "",
    "main_title": "",
    "brain_panel_title": "Brain (OASIS)\ndeformation field",
    "lung_panel_title": "Lung (DIR-LAB)\ndeformation field",
    "motion_colorbar_label": "Motion amplitude (mm)",
    "amplitude_mean_title": "Mean displacement",
    "amplitude_p95_title": "P95 displacement",
    "amplitude_gt10_title": "Voxels > 10 mm",
    "component_title": "Direction distribution",
    "bending_title": "Bending energy",
    "strain_title": "Strain norm",
    "logj_title": "SDlogJ",
    "lowpass_title": "Gaussian low-pass\nexplainability",
    "dataset_oasis": "Brain MR",
    "dataset_4dct": "Lung CT",
    "ylabel_mm": "mm",
    "ylabel_percent": "%",
    "ylabel_component_fraction": "Absolute component\nfraction",
    "ylabel_bending": "mean second-deriv\nenergy",
    "ylabel_strain": "mean Frobenius",
    "ylabel_logj": "std",
    "xlabel_sigma": "Gaussian sigma (mm)",
    "ylabel_explained_energy": "Explained energy",
    "legend_abs_ux": "|ux|",
    "legend_abs_uy": "|uy|",
    "legend_abs_uz": "|uz|",
    "legend_oasis": "Brain MR",
    "legend_4dct": "Lung CT",
    "bottom_note": "Statistics are based on 10 brain MR (OASIS) and 10 lung CT (DIR-LAB) cases using state-of-the-art registration methods.",
    "caption": "The composite contrasts smaller but locally heterogeneous OASIS brain deformation with larger, smoother respiratory LungCT motion.",
}

# Font sizes in points.
FONTS = {
    "panel_letter": 30,
    "main_title": 30,
    "panel_title": 18,
    "subplot_title": 16,
    "axis_label": 15,
    "tick_label": 12,
    "legend": 12,
    "colorbar_label": 13,
    "colorbar_tick": 11,
    "note": 11,
    "bottom_note": 14,
}

# Styling and layout controls.
STYLE = {
    "figsize": (17.5, 8.0),
    "dpi": 300,
    "save_png": True,
    "save_pdf": True,
    "save_svg": True,
    "output_stem": "deformation_regime_composite",
    "motion_cmap": "viridis",
    "oasis_color": "#1f77b4",
    "lung_color": "#ff7f0e",
    "component_colors": ["#4C78A8", "#F58518", "#54A24B"],
    "box_width": 0.55,
    "scatter_size": 24,
    "scatter_alpha": 0.85,
    "jitter": 0.035,
    "grid_alpha": 0.22,
    "spine_linewidth": 0.9,
    "component_mode": "aggregate",  # "aggregate" or "case"
    "lowpass_log2_x": True,
    "show_bottom_note": True,
    "grid_top_without_title": 0.93,
    "grid_top_with_title": 0.86,
}

# Selected methods and displayed dataset order.
SELECTED_METHODS = {
    "OASIS": "OASIS SINR cps4 alpha1000",
    "4DCT": "4DCT IDIR default",
}
DATASET_ORDER = ["OASIS", "4DCT"]

# Source files.
SOURCE_PATHS = {
    "brain_overlay_png": MOTION_ARTIFACT_DIR / "forward_grid_moving_overlay_oasis_rank009.png",
    "lung_overlay_png": MOTION_ARTIFACT_DIR / "forward_grid_moving_overlay_lungct_case08.png",
    "case_metrics_csv": STATS_DIR / "tables/case_metrics_comprehensive.csv",
    "lowpass_curves_csv": STATS_DIR / "curves/lowpass_curves.csv",
}

# Crops are fractional image coordinates: (x0, y0, x1, y1). They remove the
# rasterized source titles/colorbars so all composite text is editable here.
IMAGE_CROPS = {
    "brain": (0.18, 0.075, 0.835, 0.94),
    "lung": (0.015, 0.085, 0.825, 0.90),
}
MOTION_COLORBAR_RANGES = {
    "brain": (0.0, 8.0),
    "lung": (0.0, 25.0),
}


In [ ]:
import hashlib
import json
from datetime import datetime, timezone

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image as DisplayImage
from IPython.display import Markdown, display

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

missing = [str(path) for path in SOURCE_PATHS.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing source files:\n" + "\n".join(missing))

case_df_all = pd.read_csv(SOURCE_PATHS["case_metrics_csv"])
lowpass_df_all = pd.read_csv(SOURCE_PATHS["lowpass_curves_csv"])

SDLOGJ_SUMMARY_KEYS = [
    "regularity_fg_registration_crop.log_jdet_std_fg",
    "regularity.log_jdet_std_fg",
    "log_jdet_std_fg",
    "fg_log_jdet_std",
]

def resolve_summary_path(path_text):
    path_text = str(path_text)
    candidates = [Path(path_text)]
    if path_text.startswith("/database/"):
        candidates.append(Path("/data/hengjie") / path_text[len("/database/") :])
    if path_text.startswith("/homebase/"):
        candidates.append(Path("/home/hengjie") / path_text[len("/homebase/") :])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]

def flatten_dict(payload, prefix=""):
    flat = {}
    if isinstance(payload, dict):
        for key, value in payload.items():
            child_prefix = f"{prefix}.{key}" if prefix else str(key)
            flat.update(flatten_dict(value, child_prefix))
    else:
        flat[prefix] = payload
    return flat

def load_positive_fg_sdlogj(path_text):
    summary_path = resolve_summary_path(path_text)
    flat = flatten_dict(json.loads(summary_path.read_text()))
    for key in SDLOGJ_SUMMARY_KEYS:
        value = flat.get(key)
        if value is not None and np.isfinite(float(value)):
            return {"value": float(value), "key": key, "resolved_summary_path": str(summary_path)}
    raise KeyError(f"No positive foreground SDlogJ key found in {summary_path}")

def selected_rows(df):
    frames = []
    for dataset in DATASET_ORDER:
        method = SELECTED_METHODS[dataset]
        sub = df[(df["dataset"] == dataset) & (df["role"] == "primary") & (df["method_label"] == method)].copy()
        if sub.empty:
            raise ValueError(f"No rows found for {dataset}: {method}")
        sub["dataset_display"] = TEXT[f"dataset_{dataset.lower()}"] if dataset != "4DCT" else TEXT["dataset_4dct"]
        frames.append(sub)
    return pd.concat(frames, ignore_index=True)

case_df = selected_rows(case_df_all)
lowpass_df = selected_rows(lowpass_df_all)
sdlogj_records = [load_positive_fg_sdlogj(path) for path in case_df["summary_path"]]
case_df["sdlogj_positive_fg"] = [record["value"] for record in sdlogj_records]
case_df["sdlogj_positive_fg_source_key"] = [record["key"] for record in sdlogj_records]
case_df["resolved_summary_path"] = [record["resolved_summary_path"] for record in sdlogj_records]

case_counts = case_df.groupby("dataset")["case_index"].nunique().to_dict()
sdlogj_source_keys = sorted(set(case_df["sdlogj_positive_fg_source_key"]))
display(Markdown("Loaded selected cases: " + ", ".join(f"{k}={v}" for k, v in case_counts.items())))
display(Markdown("SDlogJ panel uses positive foreground summary keys: " + ", ".join(f"`{key}`" for key in sdlogj_source_keys)))


In [ ]:
def dataset_label(dataset):
    return TEXT["dataset_4dct"] if dataset == "4DCT" else TEXT[f"dataset_{dataset.lower()}"]

def dataset_color(dataset):
    return STYLE["lung_color"] if dataset == "4DCT" else STYLE["oasis_color"]

def style_axis(ax):
    ax.grid(True, axis="y", alpha=STYLE["grid_alpha"], linewidth=0.8)
    ax.tick_params(axis="both", labelsize=FONTS["tick_label"], width=STYLE["spine_linewidth"])
    for spine in ax.spines.values():
        spine.set_linewidth(STYLE["spine_linewidth"])

def crop_image_fraction(img, crop):
    x0, y0, x1, y1 = crop
    height, width = img.shape[:2]
    ix0, ix1 = int(round(x0 * width)), int(round(x1 * width))
    iy0, iy1 = int(round(y0 * height)), int(round(y1 * height))
    return img[iy0:iy1, ix0:ix1]

def add_motion_panel(fig, ax, image_path, crop_key, title_key):
    ax.set_anchor("C")
    img = plt.imread(image_path)
    cropped = crop_image_fraction(img, IMAGE_CROPS[crop_key])
    ax.imshow(cropped)
    ax.set_title(TEXT[title_key], loc="center", fontsize=FONTS["panel_title"], fontweight="bold", pad=8)
    ax.axis("off")
    vmin, vmax = MOTION_COLORBAR_RANGES[crop_key]
    sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=vmin, vmax=vmax), cmap=STYLE["motion_cmap"])
    sm.set_array([])
    cax = ax.inset_axes([1.015, 0.05, 0.035, 0.88])
    colorbar = fig.colorbar(sm, cax=cax)
    colorbar.set_label(TEXT["motion_colorbar_label"], fontsize=FONTS["colorbar_label"])
    colorbar.ax.tick_params(labelsize=FONTS["colorbar_tick"], width=STYLE["spine_linewidth"])
    return colorbar

def metric_boxplot(ax, metric, title_key, ylabel_key):
    rng = np.random.default_rng(1234)
    data = [case_df[case_df["dataset"] == dataset][metric].astype(float).to_numpy() for dataset in DATASET_ORDER]
    labels = [dataset_label(dataset) for dataset in DATASET_ORDER]
    bp = ax.boxplot(
        data,
        labels=labels,
        patch_artist=True,
        widths=STYLE["box_width"],
        showfliers=False,
        medianprops={"color": "black", "linewidth": 1.2},
    )
    for patch, dataset in zip(bp["boxes"], DATASET_ORDER):
        patch.set_facecolor(mpl.colors.to_rgba(dataset_color(dataset), 0.14))
        patch.set_edgecolor(dataset_color(dataset))
        patch.set_linewidth(1.2)
    for key in ("whiskers", "caps"):
        for item in bp[key]:
            item.set_linewidth(1.0)
    for x_pos, vals, dataset in zip([1, 2], data, DATASET_ORDER):
        jitter = rng.normal(0.0, STYLE["jitter"], size=len(vals))
        ax.scatter(
            np.full(len(vals), x_pos) + jitter,
            vals,
            s=STYLE["scatter_size"],
            color=dataset_color(dataset),
            alpha=STYLE["scatter_alpha"],
            edgecolors="white",
            linewidths=0.35,
            zorder=3,
        )
    ax.set_title(TEXT[title_key], fontsize=FONTS["subplot_title"], pad=8)
    ax.set_ylabel(TEXT[ylabel_key], fontsize=FONTS["axis_label"])
    style_axis(ax)

def plot_component_fractions(ax):
    components = ["abs_ux_fraction", "abs_uy_fraction", "abs_uz_fraction"]
    component_labels = [TEXT["legend_abs_ux"], TEXT["legend_abs_uy"], TEXT["legend_abs_uz"]]
    colors = STYLE["component_colors"]
    if STYLE["component_mode"] == "case":
        plot_df = case_df.sort_values(["dataset", "case_index"]).reset_index(drop=True)
        x = np.arange(len(plot_df))
        bottoms = np.zeros(len(plot_df))
        tick_labels = [f"{dataset_label(row.dataset)}\n{int(row.case_index) + 1}" for row in plot_df.itertuples()]
        for component, label, color in zip(components, component_labels, colors):
            values = plot_df[component].astype(float).to_numpy()
            ax.bar(x, values, bottom=bottoms, label=label, color=color, width=0.8)
            bottoms += values
        ax.set_xticks(x)
        ax.set_xticklabels(tick_labels, rotation=0, fontsize=max(8, FONTS["tick_label"] - 3))
    else:
        plot_df = case_df.groupby("dataset", sort=False)[components].mean().loc[DATASET_ORDER]
        x = np.arange(len(DATASET_ORDER))
        bottoms = np.zeros(len(DATASET_ORDER))
        for component, label, color in zip(components, component_labels, colors):
            values = plot_df[component].astype(float).to_numpy()
            ax.bar(x, values, bottom=bottoms, label=label, color=color, width=0.62)
            bottoms += values
        ax.set_xticks(x)
        ax.set_xticklabels([dataset_label(dataset) for dataset in DATASET_ORDER], fontsize=FONTS["tick_label"])
    ax.set_ylim(0.0, 1.0)
    ax.set_title(TEXT["component_title"], fontsize=FONTS["subplot_title"], pad=8)
    ax.set_ylabel(TEXT["ylabel_component_fraction"], fontsize=FONTS["axis_label"])
    legend = ax.legend(fontsize=FONTS["legend"], frameon=False, loc="center left", ncols=1, bbox_to_anchor=(1.04, 0.5), borderaxespad=0.0)
    style_axis(ax)

def plot_lowpass(ax):
    for dataset in DATASET_ORDER:
        sub = lowpass_df[lowpass_df["dataset"] == dataset]
        stats = sub.groupby("sigma_mm")["explained_energy"].agg(["mean", "std"]).reset_index()
        color = dataset_color(dataset)
        label = TEXT["legend_4dct"] if dataset == "4DCT" else TEXT["legend_oasis"]
        ax.plot(stats["sigma_mm"], stats["mean"], marker="o", color=color, linewidth=2.2, markersize=5.5, label=label)
        ax.fill_between(
            stats["sigma_mm"],
            stats["mean"] - stats["std"],
            stats["mean"] + stats["std"],
            color=color,
            alpha=0.14,
            linewidth=0,
        )
    if STYLE["lowpass_log2_x"]:
        ax.set_xscale("log", base=2)
        ticks = sorted(lowpass_df["sigma_mm"].unique())
        ax.set_xticks(ticks)
        ax.set_xticklabels([f"{int(t)}" for t in ticks], fontsize=FONTS["tick_label"])
    ax.set_ylim(0.0, 1.05)
    ax.set_title(TEXT["lowpass_title"], fontsize=FONTS["subplot_title"], pad=8)
    ax.set_xlabel(TEXT["xlabel_sigma"], fontsize=FONTS["axis_label"])
    ax.set_ylabel(TEXT["ylabel_explained_energy"], fontsize=FONTS["axis_label"])
    ax.legend(fontsize=FONTS["legend"], frameon=False, loc="lower left")
    ax.grid(True, alpha=STYLE["grid_alpha"], linewidth=0.8)
    ax.tick_params(axis="both", labelsize=FONTS["tick_label"], width=STYLE["spine_linewidth"])
    for spine in ax.spines.values():
        spine.set_linewidth(STYLE["spine_linewidth"])

def hash_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


In [ ]:
mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.titleweight": "bold",
    "axes.labelsize": FONTS["axis_label"],
    "xtick.labelsize": FONTS["tick_label"],
    "ytick.labelsize": FONTS["tick_label"],
    "legend.fontsize": FONTS["legend"],
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

has_main_title = bool(TEXT["main_title"].strip())
grid_top = STYLE["grid_top_with_title"] if has_main_title else STYLE["grid_top_without_title"]

fig = plt.figure(figsize=STYLE["figsize"], dpi=STYLE["dpi"], constrained_layout=False)
gs = fig.add_gridspec(
    2,
    5,
    width_ratios=[2.25, 1.0, 1.0, 1.0, 1.35],
    height_ratios=[1.0, 1.0],
    left=0.045,
    right=0.985,
    bottom=0.12,
    top=grid_top,
    wspace=0.48,
    hspace=0.58,
)

if has_main_title:
    fig.suptitle(TEXT["main_title"], fontsize=FONTS["main_title"], fontweight="bold", y=0.955)
if TEXT["panel_letter"]:
    fig.text(0.012, 0.955, TEXT["panel_letter"], fontsize=FONTS["panel_letter"], fontweight="bold", color="#0b5e7a", va="top")

ax_brain = fig.add_subplot(gs[0, 0])
ax_lung = fig.add_subplot(gs[1, 0])
add_motion_panel(fig, ax_brain, SOURCE_PATHS["brain_overlay_png"], "brain", "brain_panel_title")
add_motion_panel(fig, ax_lung, SOURCE_PATHS["lung_overlay_png"], "lung", "lung_panel_title")

metric_boxplot(fig.add_subplot(gs[0, 1]), "mag_mean_mm", "amplitude_mean_title", "ylabel_mm")
metric_boxplot(fig.add_subplot(gs[0, 2]), "mag_p95_mm", "amplitude_p95_title", "ylabel_mm")
metric_boxplot(fig.add_subplot(gs[0, 3]), "pct_gt_10mm", "amplitude_gt10_title", "ylabel_percent")
plot_component_fractions(fig.add_subplot(gs[0, 4]))

metric_boxplot(fig.add_subplot(gs[1, 1]), "bending_energy", "bending_title", "ylabel_bending")
metric_boxplot(fig.add_subplot(gs[1, 2]), "strain_norm_mean", "strain_title", "ylabel_strain")
metric_boxplot(fig.add_subplot(gs[1, 3]), "sdlogj_positive_fg", "logj_title", "ylabel_logj")
plot_lowpass(fig.add_subplot(gs[1, 4]))

if STYLE["show_bottom_note"] and TEXT["bottom_note"]:
    fig.text(0.50, 0.035, TEXT["bottom_note"], fontsize=FONTS["bottom_note"], color="#222222", ha="center")

output_paths = {}
stem = STYLE["output_stem"]
if STYLE["save_png"]:
    output_paths["png"] = ARTIFACT_DIR / f"{stem}.png"
    fig.savefig(output_paths["png"], dpi=STYLE["dpi"], bbox_inches="tight", facecolor="white")
if STYLE["save_pdf"]:
    output_paths["pdf"] = ARTIFACT_DIR / f"{stem}.pdf"
    fig.savefig(output_paths["pdf"], bbox_inches="tight", facecolor="white")
if STYLE["save_svg"]:
    output_paths["svg"] = ARTIFACT_DIR / f"{stem}.svg"
    fig.savefig(output_paths["svg"], bbox_inches="tight", facecolor="white")

manifest = {
    "artifact": "deformation_regime_composite_v001",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "repo_root": str(REPO_ROOT),
    "source_paths": {key: str(path.relative_to(REPO_ROOT)) for key, path in SOURCE_PATHS.items()},
    "selected_methods": SELECTED_METHODS,
    "dataset_order": DATASET_ORDER,
    "case_counts": {key: int(value) for key, value in case_counts.items()},
    "metric_overrides": {
        "SDlogJ panel": "Uses positive-Jacobian foreground SDlogJ loaded from per-run summary JSON, not the motion-regime CSV clipped-log column.",
        "source_keys": sdlogj_source_keys,
    },
    "text": TEXT,
    "fonts": FONTS,
    "style": STYLE,
    "image_crops": IMAGE_CROPS,
    "motion_colorbar_ranges": MOTION_COLORBAR_RANGES,
    "outputs": {key: str(path.relative_to(REPO_ROOT)) for key, path in output_paths.items()},
    "output_sha256": {key: hash_file(path) for key, path in output_paths.items()},
}
manifest_path = ARTIFACT_DIR / f"{stem}_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")

plt.close(fig)
display(Markdown(TEXT["caption"]))
display(DisplayImage(filename=str(output_paths["png"])))
display(Markdown("Saved outputs: " + ", ".join(f"`{path.name}`" for path in output_paths.values()) + f", `{manifest_path.name}`"))
